# Experiments

This file contains multiple experiments that were done in order to find the solution to the [task](task.md). Most of the necesary code is located in [solution.py](solution.py).

## Preparation

First we load and prepare the data. Preparation includes:
- removing whitespaces
- assigning labels for cheap, average and expensive houses
- converting categorical data to numerical values

In [5]:
from solution import prepare_train_data, NetConfig, NeuralNetwork
import numpy as np
import torch.nn as nn

X, y = prepare_train_data("train_data.csv")
X, y = np.array(X), np.array(y)

Below are the methods we will use to compare the quality of predictions of created models.
- `calc_accuracy` measures average accuracy across all price categories (in our case: 0, 1, 2)
- `evaluate_cv` estimates model performance using 5-fold stratified cross-validation

In [8]:
from sklearn.model_selection import StratifiedKFold
import copy

def calc_accuracy(preds, targets) -> float:
    return np.mean([
        (preds[targets == i] == i).mean()
        for i in range(3)
        if (targets == i).sum() > 0
    ])


def evaluate_cv(name: str, model, X, y) -> None:
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr, te in kf.split(X, y):
        m = copy.deepcopy(model)
        m.fit(X[tr], y[tr])
        scores.append(calc_accuracy(m.predict(X[te]), y[te]))
    
    print(f"{name}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

## Experiments

### 1. Managing unbalanced dataset
Since the task specifies that the dataset favours certain house classes and is unbalanced, we need to find a way to make our model prepared for that.

In [9]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

model1 = NeuralNetwork(NetConfig(sampler=None))
evaluate_cv("No balancing", model1, X, y)

model2 = NeuralNetwork(NetConfig(sampler=SMOTE))
evaluate_cv("SMOTE", model2, X, y)

model3 = NeuralNetwork(NetConfig(sampler=RandomUnderSampler))
evaluate_cv("RandomUnderSampler", model3, X, y)

model5 = NeuralNetwork(NetConfig(sampler=None, class_weight=True))
evaluate_cv("Class weights", model5, X, y)


No balancing: 0.7468 ± 0.0186
SMOTE: 0.8739 ± 0.0059
RandomUnderSampler: 0.8752 ± 0.0072
Class weights: 0.8782 ± 0.0060


We can see that **balancing the dataset does improve the results**. 
- Without any balancing the model scores  **0.749**, while every balancing method jumps to around **0.87–0.88**. 
- The best result comes from **Class Weight (0.8826 ± 0.0043)**, which we will use as the default for all further experiments.

### 2. Network layers and their activation

In [11]:
model1 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], activation=[nn.ReLU]))
evaluate_cv("[256,128,64] with ReLU", model1, X, y)

model2 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], activation=[nn.Identity]))
evaluate_cv("[256,128,64] with linear activation", model2, X, y)

model3 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], activation=[nn.LeakyReLU]))
evaluate_cv("[256,128,64] with LeakyReLU", model3, X, y)

model4 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], activation=[nn.Tanh]))
evaluate_cv("[256,128,64] with Tanh", model4, X, y)

model5 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], activation=[nn.ReLU, nn.ReLU, nn.LeakyReLU]))
evaluate_cv("[256,128,64] with ReLU/LeakyReLU", model5, X, y)


[256,128,64] with ReLU: 0.8854 ± 0.0068
[256,128,64] with linear activation: 0.8801 ± 0.0038
[256,128,64] with LeakyReLU: 0.8853 ± 0.0063
[256,128,64] with Tanh: 0.8856 ± 0.0061
[256,128,64] with ReLU/LeakyReLU: 0.8864 ± 0.0053


All three activation configurations perform nearly identically (around 0.885), so the choice of activation function has no meaningful impact here. We will stick with the default **ReLU** for simplicity.

### 3. Adding Batch Normalization


In [18]:
model_without_bn = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], activation=[nn.ReLU], batch_norm=False))
evaluate_cv("No batch normalization", model_without_bn, X, y)

model_with_bn = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], activation=[nn.ReLU], batch_norm=True))
evaluate_cv("Batch normalization", model_with_bn, X, y)

No batch normalization: 0.8857 ± 0.0048
Batch normalization: 0.8833 ± 0.0088


Batch Normalization slightly decreased accuracy and increased variance. We won't use it in next sections.

### 4. Different Dropouts


In [21]:
model_dropout_1 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], dropout=[0.05]))
evaluate_cv("Dropout 0.05", model_dropout_1, X, y)

model_dropout_2 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], dropout=[0.1]))
evaluate_cv("Dropout 0.1", model_dropout_2, X, y)

model_dropout_3 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], dropout=[0.2]))
evaluate_cv("Dropout 0.2", model_dropout_3, X, y)

model_dropout_4 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], dropout=[0.3]))
evaluate_cv("Dropout 0.3", model_dropout_4, X, y)

model_dropout_5 = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], dropout=[0.5]))
evaluate_cv("Dropout 0.5", model_dropout_5, X, y)

model_drop_dec = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], dropout=[0.4, 0.2, 0.1]))
evaluate_cv("Descending Dropout [0.3, 0.2, 0.1]", model_drop_dec, X, y)

model_drop_inc = NeuralNetwork(NetConfig(class_weight=True, layers=[256, 128, 64], dropout=[0.1, 0.2, 0.4]))
evaluate_cv("Ascending Dropout [0.1, 0.2, 0.3]", model_drop_inc, X, y)

Dropout 0.05: 0.8864 ± 0.0046
Dropout 0.1: 0.8860 ± 0.0039
Dropout 0.2: 0.8878 ± 0.0051
Dropout 0.3: 0.8849 ± 0.0042
Dropout 0.5: 0.8775 ± 0.0067
Descending Dropout [0.3, 0.2, 0.1]: 0.8836 ± 0.0052
Ascending Dropout [0.1, 0.2, 0.3]: 0.8874 ± 0.0035


After testing various dropout strategies, Ascending Dropout [0.1, 0.2, 0.3] was selected. It achieved a high accuracy of 0.8874 with the lowest variance (± 0.0035).

### 5. Grid Search

In [24]:
architectures = [
    [256, 128, 64],
    [512, 256, 128],
    [512, 128, 32]
]
epoch_options = [200, 300]
best_dropout = [0.1, 0.2, 0.3]

for layers in architectures:
    for epochs in epoch_options:
        model = NeuralNetwork(NetConfig(
            class_weight=True, 
            layers=layers, 
            dropout=best_dropout, 
            epochs=epochs
        ))

        name = f"Layers: {layers} Epochs: {epochs}"
        evaluate_cv(name, model, X, y)

Layers: [256, 128, 64] Epochs: 200: 0.8915 ± 0.0070
Layers: [256, 128, 64] Epochs: 300: 0.8901 ± 0.0082
Layers: [512, 256, 128] Epochs: 200: 0.8908 ± 0.0071
Layers: [512, 256, 128] Epochs: 300: 0.8915 ± 0.0060
Layers: [512, 128, 32] Epochs: 200: 0.8907 ± 0.0097
Layers: [512, 128, 32] Epochs: 300: 0.8932 ± 0.0070


The [512, 128, 32] architecture hit the best score (0.8932), showing that a wider input layer picks up more house features. Pushing the training to 300 epochs was the key, as these bigger models needed more time to fully converge with dropout.

## Conclusion
We skipped Batch Normalization because it added noise, and instead found that an "Ascending Dropout" [0.1, 0.2, 0.3] provided the best stability. The final [512, 128, 32] "funnel" architecture with 300 epochs reached 0.8932, proving that a wider input and more training time are key to capturing complex house features.